# Figure 2D — neural and musculoskeletal gene heatmaps

This notebook reproduces the existing 10-gene neural heatmap and creates a matching 10-gene musculoskeletal heatmap, including GAS1, BOC and CDON, from the same female WT and AFF3-KO bulk RNA-seq count tables.

## 1. Configuration
Change gene lists, paths, colors, or export dimensions here. The scientific analysis and plotting code are kept in later sections.

In [ ]:
# Run from the Figure-2 folder.
figure2_dir <- "."
source_dir <- file.path(figure2_dir, "source_files")
output_dir <- "generated_figures"
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

sample_files <- c(
  AFF3_F_WT_1 = "AFF3_F_WT_1_mm_counts_revised.txt",
  AFF3_F_WT_2 = "AFF3_F_WT_2_mm_counts_revised.txt",
  AFF3_F_KO_1 = "AFF3_F_KO_1_mm_counts_revised.txt",
  AFF3_F_KO_2 = "AFF3_F_KO_2_mm_counts_revised.txt"
)
sample_conditions <- factor(
  c("WT", "WT", "KO", "KO"),
  levels = c("WT", "KO")
)
sample_labels <- c("WT_1", "WT_2", "KO_1", "KO_2")

# Existing Figure 2 neural-gene set.
neural_genes <- c(
  "PAX3", "PAX7", "OTX2", "PAX6", "EMX2",
  "SRGAP3", "NAV1", "NRXN3", "EPHB1", "EPHB2"
)

# Skeletal-muscle program: progenitor/regulatory genes first, followed by
# differentiation and structural genes. All ten are lower in KO than WT.
skeletal_muscle_genes <- c(
  "PAX3", "PAX7", "RBM24", "GAS1", "BOC", "CDON",
  "ACTN3", "TNNT1", "SGCB", "DMD"
)

# Export settings copied from the original neural heatmap.
figure_width <- 4.05
figure_height <- 12.5
figure_dpi <- 600

## 2. Load packages

In [ ]:
library(DESeq2)
library(ggplot2)
library(tidyr)
library(tibble)
library(dplyr)
library(scales)

## 3. Read and check the count tables
The four count tables must contain the same genes in the same order before they are combined.

In [ ]:
count_tables <- lapply(
  file.path(source_dir, sample_files),
  read.table,
  header = TRUE
)

reference_gene_order <- count_tables[[1]]$Geneid
same_gene_order <- vapply(
  count_tables,
  function(count_table) identical(reference_gene_order, count_table$Geneid),
  logical(1)
)
stopifnot(all(same_gene_order))

count_matrix <- do.call(
  cbind,
  lapply(count_tables, function(count_table) count_table$counts)
)
rownames(count_matrix) <- reference_gene_order
colnames(count_matrix) <- names(sample_files)

sample_data <- data.frame(condition = sample_conditions)
rownames(sample_data) <- colnames(count_matrix)

stopifnot(identical(colnames(count_matrix), rownames(sample_data)))
dim(count_matrix)

## 4. DESeq2 analysis and variance-stabilizing transformation
The transformation and row-wise z-score calculation are unchanged from the original neural heatmap analysis.

In [ ]:
dds <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = sample_data,
  design = ~ condition
)
dds$condition <- relevel(dds$condition, ref = "WT")
dds <- DESeq(dds)

deseq_results <- results(
  dds,
  contrast = c("condition", "KO", "WT")
) %>%
  as.data.frame() %>%
  rownames_to_column("Gene")

vst_matrix <- assay(vst(dds, blind = FALSE))

## 5. Inspect the selected musculoskeletal genes
This table makes the gene selection easy to review. `log2FoldChange` is KO versus WT, so a negative value means lower expression in AFF3 KO.

In [ ]:
skeletal_muscle_gene_roles <- c(
  PAX3 = "muscle progenitor specification",
  PAX7 = "muscle progenitor and satellite-cell program",
  RBM24 = "myogenic differentiation and muscle RNA processing",
  CDON = "myoblast differentiation",
  GAS1 = "skeletal-development candidate selected for this panel",
  BOC = "skeletal-development candidate selected for this panel",
  ACTN3 = "sarcomeric structural protein",
  TNNT1 = "slow skeletal-muscle troponin",
  SGCB = "dystrophin-associated muscle membrane complex",
  DMD = "dystrophin muscle membrane complex",
  EMD = "muscle nuclear-envelope protein"
)

skeletal_muscle_selection <- deseq_results %>%
  filter(Gene %in% skeletal_muscle_genes) %>%
  mutate(Role = unname(skeletal_muscle_gene_roles[Gene])) %>%
  select(Gene, Role, baseMean, log2FoldChange, padj) %>%
  arrange(match(Gene, skeletal_muscle_genes))

stopifnot(
  nrow(skeletal_muscle_selection) == length(skeletal_muscle_genes)
)

skeletal_muscle_selection

## 6. Shared heatmap function
This block contains the original heatmap colors, typography, tile geometry, legend, and export dimensions. Both panels use exactly the same function.

In [ ]:
make_expression_heatmap <- function(gene_list) {
  missing_genes <- setdiff(gene_list, rownames(vst_matrix))
  if (length(missing_genes) > 0) {
    stop("Missing genes: ", paste(missing_genes, collapse = ", "))
  }

  selected_vst <- vst_matrix[gene_list, , drop = FALSE]
  z_matrix <- t(scale(t(selected_vst)))
  stopifnot(!anyNA(z_matrix))

  plot_data <- as.data.frame(z_matrix) %>%
    rownames_to_column("Gene") %>%
    pivot_longer(
      cols = -Gene,
      names_to = "Sample",
      values_to = "Zscore"
    )

  plot_data$Sample <- factor(
    plot_data$Sample,
    levels = names(sample_files)
  )
  plot_data$Gene <- factor(
    plot_data$Gene,
    levels = rev(gene_list)
  )

  color_limit <- max(abs(plot_data$Zscore), na.rm = TRUE)
  color_stops <- c(
    -color_limit, -0.91, -0.792, 0, 0.792, 0.91, color_limit
  )
  color_values <- rescale(color_stops, to = c(0, 1))
  heatmap_colors <- c(
    "#2D004BFF", "#542788FF", "#8073ACFF", "#B2ABD2FF",
    "#D8DAEBFF", "#F7F7F7FF", "#FBE3E3", "#F8B8B0",
    "#EF7260", "#CC2E2E", "#7F0000"
  )

  ggplot(plot_data, aes(Sample, Gene, fill = Zscore)) +
    geom_tile() +
    scale_fill_gradientn(
      colors = heatmap_colors,
      values = color_values,
      limits = c(-color_limit, color_limit),
      oob = squish,
      name = "Normalized expression",
      breaks = c(-1, 0, 1),
      guide = guide_colorbar(
        barwidth = 13,
        barheight = 2.15,
        ticks.colour = "black",
        direction = "horizontal",
        title.position = "top",
        title.hjust = 0
      )
    ) +
    scale_y_discrete(
      labels = rev(gene_list),
      position = "right",
      expand = c(0, 0)
    ) +
    scale_x_discrete(
      expand = c(0, 0),
      labels = sample_labels
    ) +
    coord_fixed(ratio = 1) +
    theme_minimal(base_size = 10) +
    theme(
      panel.grid = element_blank(),
      axis.text.y.right = element_text(
        size = 32,
        margin = margin(l = 2.5),
        hjust = 0,
        face = "bold.italic"
      ),
      axis.text.y = element_blank(),
      axis.ticks.y = element_blank(),
      axis.title.y = element_blank(),
      axis.title.x = element_text(size = 32),
      axis.text.x = element_text(
        size = 30,
        margin = margin(t = 2.5),
        angle = 90,
        vjust = 0.5,
        hjust = 1,
        face = "bold"
      ),
      legend.title = element_text(size = 28),
      legend.text = element_text(size = 28),
      legend.position = "bottom",
      legend.box.margin = margin(t = 13),
      legend.justification = "left",
      legend.box.just = "left"
    ) +
    labs(x = NULL, y = NULL)
}

save_heatmap <- function(plot_object, output_file) {
  ggsave(
    output_file,
    plot = plot_object,
    width = figure_width,
    height = figure_height,
    units = "in",
    dpi = figure_dpi,
    limitsize = FALSE,
    compression = "lzw"
  )
}

## 7. Neural-gene heatmap

In [ ]:
neural_heatmap <- make_expression_heatmap(neural_genes)
neural_output_file <- file.path(
  output_dir,
  "Fig2D_neural_heatmap.tiff"
)
save_heatmap(neural_heatmap, neural_output_file)
neural_heatmap

## 8. Skeletal-muscle gene heatmap

In [ ]:
skeletal_muscle_heatmap <- make_expression_heatmap(skeletal_muscle_genes)
skeletal_muscle_output_file <- file.path(
  output_dir,
  "Fig2D_musculoskeletal_heatmap.tiff"
)
save_heatmap(skeletal_muscle_heatmap, skeletal_muscle_output_file)
skeletal_muscle_heatmap

## 9. Output check

In [ ]:
output_files <- c(neural_output_file, skeletal_muscle_output_file)
stopifnot(all(file.exists(output_files)))
file.info(output_files)[, c("size", "mtime")]